# Observation Inspector
Step through the environment manually and inspect every SDF channel the network sees.

In [1]:
import sys, math, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display, clear_output

sys.path.insert(0, str(Path('scripts').resolve()))
sys.path.insert(0, str(Path('crates/python/python').resolve()))
from u_nesting_gym import UNestingGymEnv

# ── Parameters ────────────────────────────────────────────────────────────────
JSON        = 'data/aligner_svgs.json'
PLATE_W     = 150.0
PLATE_H     = 150.0
N_PARTS     = 15
N_ROTATIONS = 8
SEED        = 42
SDF_CMAP    = 'RdYlGn'   # green=far from obstacle, red=close
# ──────────────────────────────────────────────────────────────────────────────

env = UNestingGymEnv(JSON, plate_width=PLATE_W, plate_height=PLATE_H)
rotations_rad = env.get_rotation_angles(N_ROTATIONS)
rot_deg = [round(math.degrees(r)) for r in rotations_rad]

rng = random.Random(SEED)
lib_ids = env.sample_episode_ids(n=N_PARTS, rng=rng)
env.reset(lib_ids)

geom_ids = [env._episode_geoms()[i]['id'] for i in range(N_PARTS)]
print(f'Episode parts: {geom_ids}')
print(f'Rotations (deg): {rot_deg}')

Episode parts: ['AZ014', 'AZ003', 'AZ035', 'AZ031', 'AZ028', 'AZ017', 'AZ013', 'AZ011', 'AZ054', 'AZ002', 'AZ001', 'AZ005', 'AZ065', 'AZ071', 'AZ032']
Rotations (deg): [0, 45, 90, 135, 180, 225, 270, 315]


## Step through the episode
Choose a part and rotation, then click **Place**. Use **Undo** to go back.

In [2]:

# ── Widgets ───────────────────────────────────────────────────────────────────
remaining_ids = list(range(N_PARTS))
placement_history = []   # list of (part_id, rot_id)

w_part = widgets.Dropdown(
    options=[(f'{i}: {geom_ids[i]}', i) for i in range(N_PARTS)],
    description='Part:',
    layout=widgets.Layout(width='280px'),
)
w_rot = widgets.Dropdown(
    options=[(f'{d}°', i) for i, d in enumerate(rot_deg)],
    description='Rotation:',
    layout=widgets.Layout(width='180px'),
)
btn_place = widgets.Button(description='Place', button_style='success', layout=widgets.Layout(width='100px'))
btn_undo  = widgets.Button(description='Undo',  button_style='warning', layout=widgets.Layout(width='100px'))
btn_reset = widgets.Button(description='Reset', button_style='danger',  layout=widgets.Layout(width='100px'))
out_main  = widgets.Output()

def _update_part_dropdown():
    remaining = env.remaining_item_ids()
    w_part.options = [(f'{i}: {geom_ids[i]}', i) for i in remaining]

def _show():
    remaining = env.remaining_item_ids()
    obs_np, scalars_np = env.preview_images_per_rotation(rotations_rad)
    N, two_Rp1, H, W = obs_np.shape
    R = (two_Rp1 - 1) // 2

    part_id = w_part.value if w_part.value is not None else (remaining[0] if remaining else 0)
    rot_id  = w_rot.value

    board_sdf    = obs_np[part_id, 0]                    # (H, W)
    after_sdf    = obs_np[part_id, rot_id + 1]           # (H, W)
    isolated_sdf = obs_np[part_id, R + 1 + rot_id]       # (H, W)
    delta        = after_sdf - board_sdf                  # always <= 0

    with out_main:
        clear_output(wait=True)

        # ── Row 1: the 4 key channels for selected (part, rotation) ──────────
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        fig.suptitle(
            f'Step {N_PARTS - len(remaining)}/{N_PARTS}  |  '
            f'Selected: part {part_id} ({geom_ids[part_id]}) @ {rot_deg[rot_id]}°  |  '
            f'Placed: {env.n_placed()}  density={env.packing_density():.3f}',
            fontsize=11, fontweight='bold'
        )

        im0 = axes[0].imshow(board_sdf, cmap=SDF_CMAP, vmin=0, vmax=1, origin='lower')
        axes[0].set_title('Channel 0: board SDF\n(green=far from wall/part, red=close)', fontsize=9)
        plt.colorbar(im0, ax=axes[0], fraction=0.046)

        im1 = axes[1].imshow(after_sdf, cmap=SDF_CMAP, vmin=0, vmax=1, origin='lower')
        axes[1].set_title(f'Channel {rot_id+1}: after placing @ {rot_deg[rot_id]}°\n(what board looks like after this choice)', fontsize=9)
        plt.colorbar(im1, ax=axes[1], fraction=0.046)

        im2 = axes[2].imshow(isolated_sdf, cmap=SDF_CMAP, vmin=0, vmax=1, origin='lower')
        axes[2].set_title(f'Channel {R+1+rot_id}: isolated part shape @ {rot_deg[rot_id]}°\n(centered, no board context)', fontsize=9)
        plt.colorbar(im2, ax=axes[2], fraction=0.046)

        # Delta is always <= 0: placing a part only reduces SDF values (new obstacle added).
        # Scale to [min, 0] so the full colormap is used: green=0 (no change), red=large drop.
        dmin = float(delta.min())
        vmin_d = min(dmin, -0.01)  # avoid degenerate range when part doesn't fit
        im3 = axes[3].imshow(delta, cmap=SDF_CMAP, vmin=vmin_d, vmax=0, origin='lower')
        axes[3].set_title('Delta (after − board)\ngreen=no change, red=SDF reduced (part footprint)', fontsize=9)
        plt.colorbar(im3, ax=axes[3], fraction=0.046)

        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

        # ── Row 2: all 8 rotations side by side ──────────────────────────────
        fig2, axes2 = plt.subplots(3, R, figsize=(2.2*R, 7))
        fig2.suptitle(f'All {R} rotations for part {part_id} ({geom_ids[part_id]})', fontsize=10)

        for r in range(R):
            highlight = (r == rot_id)

            # Row 0: after-SDF
            ax = axes2[0, r]
            ax.imshow(obs_np[part_id, r+1], cmap=SDF_CMAP, vmin=0, vmax=1, origin='lower')
            ax.set_title(f'{rot_deg[r]}°{" ←" if highlight else ""}', fontsize=8,
                        color='goldenrod' if highlight else 'black',
                        fontweight='bold' if highlight else 'normal')
            for spine in ax.spines.values():
                spine.set_edgecolor('gold' if highlight else 'none')
                spine.set_linewidth(4 if highlight else 0)
            ax.set_xticks([]); ax.set_yticks([])

            # Row 1: isolated shape
            ax = axes2[1, r]
            ax.imshow(obs_np[part_id, R+1+r], cmap=SDF_CMAP, vmin=0, vmax=1, origin='lower')
            ax.set_title('shape', fontsize=7)
            ax.set_xticks([]); ax.set_yticks([])

            # Row 2: delta (white=no change, red=part footprint)
            ax = axes2[2, r]
            d = obs_np[part_id, r+1] - obs_np[part_id, 0]
            dmin_r = float(d.min())
            ax.imshow(d, cmap=SDF_CMAP, vmin=min(dmin_r, -0.01), vmax=0, origin='lower')
            sc = scalars_np[part_id, r]
            ax.set_title(f'a={sc[0]:.2f} w={sc[1]:.2f} h={sc[2]:.2f}', fontsize=6)
            ax.set_xticks([]); ax.set_yticks([])

        axes2[0,0].set_ylabel('after SDF', fontsize=8)
        axes2[1,0].set_ylabel('isolated shape', fontsize=8)
        axes2[2,0].set_ylabel('delta', fontsize=8)
        plt.tight_layout()
        plt.show()

        # ── Row 3: all parts × selected rotation ─────────────────────────────
        fig3, axes3 = plt.subplots(2, N_PARTS, figsize=(2.2*N_PARTS, 5))
        fig3.suptitle(f'All {N_PARTS} parts at rotation {rot_deg[rot_id]}° — top: after SDF, bottom: isolated shape', fontsize=10)
        for i in range(N_PARTS):
            is_remaining = i in remaining
            is_selected  = i == part_id
            alpha = 1.0 if is_remaining else 0.3

            ax = axes3[0, i]
            ax.imshow(obs_np[i, rot_id+1], cmap=SDF_CMAP, vmin=0, vmax=1, origin='lower', alpha=alpha)
            label = f'{i}\n{geom_ids[i][:6]}'
            color = 'goldenrod' if is_selected else ('black' if is_remaining else 'gray')
            ax.set_title(label, fontsize=6, color=color,
                        fontweight='bold' if is_selected else 'normal')
            ax.set_xticks([]); ax.set_yticks([])
            if not is_remaining:
                ax.text(0.5, 0.5, 'placed', ha='center', va='center',
                       transform=ax.transAxes, fontsize=7, color='gray')

            ax2 = axes3[1, i]
            ax2.imshow(obs_np[i, R+1+rot_id], cmap=SDF_CMAP, vmin=0, vmax=1, origin='lower', alpha=alpha)
            sc = scalars_np[i, rot_id]
            ax2.set_title(f'a={sc[0]:.1f}', fontsize=6)
            ax2.set_xticks([]); ax2.set_yticks([])

        plt.tight_layout()
        plt.show()

def on_place(_):
    part_id = w_part.value
    rot_id  = w_rot.value
    if part_id is None:
        return
    success = env.place_with_rotation(part_id, rotations_rad[rot_id])
    if success:
        placement_history.append((part_id, rot_id))
    _update_part_dropdown()
    _show()

def on_undo(_):
    if not placement_history:
        return
    placement_history.pop()
    env.reset(lib_ids)
    for pid, rid in placement_history:
        env.place_with_rotation(pid, rotations_rad[rid])
    _update_part_dropdown()
    _show()

def on_reset(_):
    placement_history.clear()
    env.reset(lib_ids)
    _update_part_dropdown()
    _show()

def on_change(_):
    _show()

btn_place.on_click(on_place)
btn_undo.on_click(on_undo)
btn_reset.on_click(on_reset)
w_part.observe(on_change, names='value')
w_rot.observe(on_change,  names='value')

display(
    widgets.HBox([w_part, w_rot, btn_place, btn_undo, btn_reset]),
    out_main
)
_show()


Output()